# Practical 7: Regression and ANOVA

**Dataset:** TeachingRatings.csv

**Aim:** Fit simple regression models and a one-way ANOVA, then interpret the coefficients and p-values.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

ratings = pd.read_csv("TeachingRatings.csv")

def fit_line(x, y):
    fit = stats.linregress(x, y)
    n = len(x)
    r2 = fit.rvalue ** 2
    f_stat = r2 / (1 - r2) * (n - 2)
    return pd.DataFrame({
        "Coefficient": [fit.intercept, fit.slope],
        "Std error": [fit.intercept_stderr, fit.stderr],
    }, index=["Intercept", "Slope"]).round(4), fit, r2, f_stat

print(ratings.shape)

(463, 13)


### Model 1: eval ~ gender

Gender is coded as male = 1 and female = 0. The slope is the difference in average evaluation between men and women.

H0: slope = 0. H1: slope is not 0.

In [2]:
male_flag = (ratings["gender"] == "male").astype(int)
table1, fit1, r2_1, f1 = fit_line(male_flag, ratings["eval"])
print(table1)
print("R-squared  :", round(r2_1, 4))
print("F-statistic:", round(f1, 4))
print("p-value    :", round(fit1.pvalue, 5))
verdict1 = "reject H0" if fit1.pvalue < 0.05 else "fail to reject H0"

           Coefficient  Std error
Intercept        3.901     0.0393
Slope            0.168     0.0517
R-squared  : 0.0224
F-statistic: 10.5621
p-value    : 0.00124


**Result:** The p-value 0.00124 is below 0.05, so we reject H0. Gender significantly affects evaluation score. Men score about 0.168 points higher than women, but R-squared is only 0.022, so gender explains a small part of the variation.

### Model 2: one-way ANOVA, beauty ~ age group

Age groups: up to 40 (Young), 41 to 50 (Middle), above 50 (Old).

H0: mean beauty is equal in all groups. H1: at least one mean differs.

In [3]:
ratings["age_group"] = pd.cut(ratings["age"], bins=[0, 40, 50, 100], labels=["Young", "Middle", "Old"])
parts = [g["beauty"].to_numpy() for _, g in ratings.groupby("age_group", observed=True)]
grand = ratings["beauty"].mean()
ss_between = sum(len(p) * (p.mean() - grand) ** 2 for p in parts)
ss_within = sum(((p - p.mean()) ** 2).sum() for p in parts)
df_between = len(parts) - 1
df_within = len(ratings) - len(parts)
f2, p2 = stats.f_oneway(*parts)
anova = pd.DataFrame({
    "df": [df_between, df_within],
    "sum_sq": [ss_between, ss_within],
    "mean_sq": [ss_between / df_between, ss_within / df_within],
    "F": [f2, np.nan],
    "p-value": [p2, np.nan],
}, index=["Age group", "Residual"])
print(anova)
verdict2 = "reject H0" if p2 < 0.05 else "fail to reject H0"

            df      sum_sq   mean_sq          F       p-value
Age group    2   17.800717  8.900359  15.189048  4.094085e-07
Residual   460  269.547179  0.585972        NaN           NaN


**Result:** F = 15.1890 and the p-value 4.09e-07 is below 0.05, so we reject H0. Beauty scores differ significantly between the age groups.

### Model 3: eval ~ beauty

H0: slope = 0. H1: slope is not 0.

In [4]:
table3, fit3, r2_3, f3 = fit_line(ratings["beauty"], ratings["eval"])
print(table3)
print("R-squared  :", round(r2_3, 4))
print("F-statistic:", round(f3, 4))
print("p-value    :", fit3.pvalue)
verdict3 = "reject H0" if fit3.pvalue < 0.05 else "fail to reject H0"

           Coefficient  Std error
Intercept       3.9983     0.0253
Slope           0.1330     0.0322
R-squared  : 0.0357
F-statistic: 17.0847
p-value    : 4.2471155077860486e-05


**Result:** The beauty coefficient is 0.1330 and the p-value 4.25e-05 is below 0.05, so we reject H0. Each one unit rise in beauty score raises the evaluation by about 0.133 points. R-squared is 0.036, so the relationship is significant but weak.